# M1 — Crawl arXiv bằng S3 + export (Colab CPU-only)

**Cách dùng:** (1) thêm Colab Secrets `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `ARXIV_CONTACT_EMAIL` (và `S2_API_KEY` nếu bật references) với *Notebook access* bật; (2) *Runtime → Run all* để xem kế hoạch (chưa tải gì); (3) đặt `RUN = True` ở cell 1 rồi *Run all* lại. Ngắt phiên thì chạy lại y nguyên: tiến độ được lưu trên Drive.

**Luồng:** `discover` (metadata, số phiên bản) → song song: **S3** lấy bản mới nhất của bài chưa bị sửa sau thời điểm tạo gói · **arXiv** (3 s, một kết nối) lấy các bản cũ và bản mới nhất của bài đã bị sửa · **Semantic Scholar** lấy references → kiểm chứng S3 = export trên vài bài → báo cáo.

Bucket S3 là *Requester Pays* (khoảng $0,09/GB, giá niêm yết). Đề M1 nói lấy từng phiên bản qua `export.arxiv.org`; dùng S3 cần nhóm/giảng viên đồng ý (`USE_S3 = False` quay về crawler gốc).

In [ ]:
# 1. Cấu hình — chỉ cần sửa PRESET và RUN
PRESET = "pilot"        # "pilot": 300 ID, ~3 gói S3 (~$0,14), 60 phiên bản arXiv mỗi lần chạy | "full": 20.000 ID, ~77 GB (~$6,9)
RUN = False             # False: chỉ in kế hoạch. True: chạy thật
USE_S3 = True           # False: không dùng S3, lấy mọi phiên bản qua export
REFERENCES = False      # True: lấy references (Semantic Scholar; nên có S2_API_KEY)
DRIVE_DIR = f"/content/drive/MyDrive/m1_{PRESET}"   # nơi lưu tiến độ; None = không dùng Drive
WORKERS_S3 = 3          # số gói S3 đọc song song

# --- nâng cao ---
RANGES = {
    "thien": ("2606.30734", "2607.03693"), "tien": ("2606.25734", "2606.30733"),
    "hieu": ("2606.05734", "2606.10733"), "dang": ("2604.20535", "2604.25534"),
    "pilot": ("2606.05734", "2606.06033"),
}
PRESETS = {"pilot": (["pilot"], 60), "full": (["thien", "tien", "hieu", "dang"], None)}
SELECT, ARXIV_BUDGET = PRESETS[PRESET]
STALE_MARGIN_DAYS = 1   # bản mới nhất sửa sau (timestamp gói - margin) => lấy từ arXiv
MAX_GB = 80.0           # chặn cứng tổng dung lượng S3
VERIFY_SAMPLES = 5      # số bài S3 đối chiếu với export ở cuối (mỗi bài ~1 request arXiv)
S3_ATTEMPTS, ARXIV_PART_EVERY, USD_PER_GB = 3, 500, 0.09

import os
REPO_URL = "https://github.com/hcmus-arxiv-nhom1/CodeRepo"
REPO_DIR = os.environ.get("M1_REPO_DIR", "/content/CodeRepo")
OUT_ROOT = os.environ.get("M1_OUT_ROOT", "/content/m1")
LOCAL_TAR_DIR = os.environ.get("M1_LOCAL_TAR_DIR")   # TAR có sẵn (kiểm thử không tốn phí)
SKIP_PIP = bool(os.environ.get("M1_SKIP_PIP"))
BUCKET, REGION, MANIFEST_KEY = "arxiv", "us-east-1", "src/arXiv_src_manifest.xml"

In [ ]:
# 2. Cài đặt: Drive, mã nguồn dự án (chỉ đọc), thư viện
import subprocess, sys
if DRIVE_DIR and not os.path.ismount("/content/drive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        pass                                     # không phải Colab
if not os.path.isdir(os.path.join(REPO_DIR, "src")):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
if not SKIP_PIP:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "boto3", "-r", os.path.join(REPO_DIR, "src", "requirements.txt")], check=True)
sys.path.insert(0, REPO_DIR)
print("repo commit:", subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# 3. Thư viện, credentials, hàm chung (không cần sửa)
import hashlib, io, json, re, shutil, tarfile, tempfile, threading, time
import xml.etree.ElementTree as ET
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta, timezone
from pathlib import Path, PurePosixPath

import boto3, psutil
from botocore.config import Config
from src import config
from src.crawler.discovery import discover, read_metadata
from src.crawler.downloader import download_version
from src.crawler.extract import CorruptArchiveError, extract_stream
from src.crawler.http_client import HttpClient
from src.crawler.ratelimit import get_arxiv_limiter
from src.crawler.worklock import WorkDirLock
from src.run_pipeline import (FINAL_STATUSES, JsonlWriter, SUSPICIOUS_KINDS, is_done, load_version_state,
                              process_version, summarize, version_dir, write_failed_csv)
from src.utils.ids import iter_id_range, to_folder_name

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

_ak, _sk = _secret("AWS_ACCESS_KEY_ID"), _secret("AWS_SECRET_ACCESS_KEY")
if USE_S3 and RUN and not (_ak and _sk) and not os.environ.get("AWS_PROFILE") and not os.path.exists(os.path.expanduser("~/.aws")):
    raise SystemExit("Thiếu AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY trong Colab Secrets (bật Notebook access).")
_kw = {"aws_access_key_id": _ak, "aws_secret_access_key": _sk} if (_ak and _sk) else {}
s3 = boto3.client("s3", region_name=REGION, config=Config(retries={"max_attempts": 5, "mode": "standard"}), **_kw)
del _ak, _sk, _kw

KEEP = frozenset(config.KEEP_EXTENSIONS)
now = lambda: datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
NEW_ID = re.compile(r"^(\d{4})\.(\d{4,5})$")
def id_key(i):
    m = NEW_ID.match(i)
    return (int(m.group(1)), int(m.group(2))) if m else None
def parse_ts(s):
    return datetime.fromisoformat(s.strip().replace("Z", "+00:00"))
def latest_date(meta):
    return parse_ts(meta["revised_dates"][-1] if meta["revised_dates"] else meta["submission_date"])
def pkg_ts(r):
    return datetime.strptime(r["timestamp"], "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)   # giả định UTC
def tree(root):
    return {p.relative_to(root).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(Path(root).rglob("*")) if p.is_file()}

class LockedWriter:
    def __init__(self, path):
        self.w, self.lock = JsonlWriter(path), threading.Lock()
    def write(self, rec):
        with self.lock:
            self.w.write(rec)
    def close(self):
        self.w.close()

class HashReader:
    def __init__(self, raw):
        self.raw, self.h = raw, hashlib.md5()
    def read(self, size=-1):
        b = self.raw.read(size)
        self.h.update(b)
        return b

def drive_sync_work(work):
    if DRIVE_DIR:
        dst = Path(DRIVE_DIR) / "work"; dst.mkdir(parents=True, exist_ok=True)
        for p in work.iterdir():
            if p.is_file() and p.suffix in (".jsonl", ".json", ".csv"):
                shutil.copy2(p, dst / p.name)

def drive_part(name, paths, data_root):
    if not DRIVE_DIR or not paths:
        return
    parts = Path(DRIVE_DIR) / "parts"; parts.mkdir(parents=True, exist_ok=True)
    tmp = parts / (name + ".tar.tmp")
    with tarfile.open(tmp, "w") as tf:
        for p in paths:
            if Path(p).exists():
                tf.add(p, arcname=Path(p).relative_to(data_root).as_posix())
    tmp.replace(parts / (name + ".tar"))

def drive_restore(work, data_root):
    if not DRIVE_DIR or not (Path(DRIVE_DIR) / "work").exists():
        return 0
    for p in (Path(DRIVE_DIR) / "work").iterdir():
        if not (work / p.name).exists():
            shutil.copy2(p, work / p.name)
    n = 0
    for part in sorted((Path(DRIVE_DIR) / "parts").glob("*.tar")):
        with tarfile.open(part) as tf:
            tf.extractall(data_root, filter="data"); n += 1
    return n

class Sampler(threading.Thread):
    def __init__(self, path, every=5.0):
        super().__init__(daemon=True); self.path, self.every, self.stop = path, every, threading.Event()
        self.rss, self.disk = [], []
    def run(self):
        proc = psutil.Process()
        while not self.stop.wait(self.every):
            self.rss.append(proc.memory_info().rss); self.disk.append(shutil.disk_usage(self.path).used)
    def stats(self):
        return {"ram_max_mb": round(max(self.rss, default=0) / 1e6, 1), "ram_mean_mb": round(sum(self.rss) / max(len(self.rss), 1) / 1e6, 1),
                "disk_used_max_gb": round(max(self.disk, default=0) / 1e9, 2)}

def s3_status(res, vdir, rec):
    # Cùng quy tắc process_version: mọi .tex đều 0 byte => no_tex, vN/ rỗng.
    tex = [f for f in res.files if f.lower().endswith(".tex")]
    if tex and all((vdir / f).stat().st_size == 0 for f in tex):
        shutil.rmtree(vdir); vdir.mkdir()
        rec.update(files=[], bytes_after=0, empty_tex_files=tex)
        return "no_tex"
    return "ok" if res.has_tex else "no_tex"

# kiểm tra offline
_b = io.BytesIO()
with tarfile.open(fileobj=_b, mode="w:gz") as _t:
    for _n, _x in (("m.tex", b""), ("f.png", b"png")):
        _i = tarfile.TarInfo(_n); _i.size = len(_x); _t.addfile(_i, io.BytesIO(_x))
with tempfile.TemporaryDirectory() as _d:
    _v = Path(_d) / "v1"; _r = extract_stream(io.BytesIO(_b.getvalue()), _v, KEEP, "x.gz", 1); _rec = {}
    assert s3_status(_r, _v, _rec) == "no_tex" and _v.is_dir() and not list(_v.iterdir())
print("Sẵn sàng.")

In [ ]:
# 4. Kế hoạch: manifest -> ID -> gói S3 -> chi phí (đọc manifest ~5 MB, phí không đáng kể)
try:
    body = s3.get_object(Bucket=BUCKET, Key=MANIFEST_KEY, RequestPayer="requester")["Body"].read()
except Exception as e:
    raise SystemExit(f"Không đọc được manifest S3 ({type(e).__name__}): kiểm tra AWS_ACCESS_KEY_ID/AWS_SECRET_ACCESS_KEY trong Colab Secrets và quyền s3:GetObject.")
rows = [{c.tag: (c.text or "").strip() for c in e} for e in ET.fromstring(body).iter("file")]
last_num = {}
for r in rows:
    k = id_key(r["last_item"]) if r["last_item"] else None
    if k:
        last_num[f"{k[0]:04d}"] = max(last_num.get(f"{k[0]:04d}", 0), k[1])
ids = list(dict.fromkeys(i for name in SELECT for i in iter_id_range(*RANGES[name], last_num)))
idset = set(ids)
def overlaps(r):
    a, b = id_key(r["first_item"]), id_key(r["last_item"])
    return bool(a and b) and any(a <= id_key(RANGES[n][1]) and b >= id_key(RANGES[n][0]) for n in SELECT)
pkgs = [r for r in rows if overlaps(r)] if USE_S3 else []
pkg_gb = sum(int(r["size"]) for r in pkgs) / 1e9
assert pkg_gb <= MAX_GB, f"vượt MAX_GB={MAX_GB}"
print(f"PRESET={PRESET}: {len(ids)} ID | S3: {len(pkgs)} gói, {pkg_gb:.1f} GB, ~${pkg_gb * USD_PER_GB:.2f} (giá niêm yết) | arXiv mỗi lần chạy: "
      f"{'không giới hạn' if ARXIV_BUDGET is None else str(ARXIV_BUDGET) + ' phiên bản'}")
print("=> Đặt RUN = True ở cell 1 và Run all để chạy." if not RUN else "=> RUN=True: bắt đầu chạy.")

In [ ]:
# 5. Luồng S3 và luồng arXiv (không cần sửa)
out = Path(OUT_ROOT); data_root, work = out / "corpus", out / "work"
state_lock, stop = threading.Lock(), threading.Event()

def s3_package(r, found, state, ver_log, pkg_log):
    key, ts = r["filename"], pkg_ts(r)
    a, b = id_key(r["first_item"]), id_key(r["last_item"])
    plan, staled = {}, 0
    for n in range(a[1], b[1] + 1):
        pid = f"{a[0]:04d}.{n:05d}"
        if pid not in idset or pid not in found:
            continue
        with state_lock:
            if is_done(state.get((pid, found[pid])), data_root):
                continue
        if latest_date(read_metadata(data_root, pid)) > ts - timedelta(days=STALE_MARGIN_DAYS):
            staled += 1
            continue
        plan[pid] = found[pid]
    info = {"package": key, "wanted": len(plan), "stale_skipped": staled}
    if not plan:
        info["status"] = "nothing_to_do"
        pkg_log.write({**info, "finished_at": now()}); return info
    for attempt in range(1, S3_ATTEMPTS + 1):
        t0, created, recs = time.perf_counter(), [], []
        try:
            lt = Path(LOCAL_TAR_DIR) / Path(key).name if LOCAL_TAR_DIR else None
            src = open(lt, "rb") if lt and lt.exists() else s3.get_object(Bucket=BUCKET, Key=key, RequestPayer="requester")["Body"]
            hr = HashReader(src)
            with tarfile.open(fileobj=hr, mode="r|") as tf:
                for m in tf:
                    pid = PurePosixPath(m.name).stem
                    if not m.isreg() or pid not in plan:
                        continue
                    ver, vdir = plan[pid], version_dir(data_root, pid, plan[pid])
                    rec = {"arxiv_id": pid, "version": ver, "source": "s3", "s3_package": key, "archive_name": PurePosixPath(m.name).name}
                    s = time.perf_counter()
                    try:
                        res = extract_stream(tf.extractfile(m), vdir, KEEP, PurePosixPath(m.name).name, m.size)
                    except CorruptArchiveError:
                        shutil.rmtree(vdir, ignore_errors=True)
                        continue                     # lượt vét của luồng arXiv sẽ lấy lại
                    if res.kind in SUSPICIOUS_KINDS:
                        shutil.rmtree(vdir, ignore_errors=True)
                        continue
                    created.append(vdir)
                    rec.update(kind=res.kind, files=res.files, archive_bytes=res.archive_bytes, bytes_before=res.bytes_before,
                               bytes_after=res.bytes_after, skipped_unsafe=res.skipped_unsafe, dropped=res.dropped,
                               total_s=round(time.perf_counter() - s, 3), finished_at=now())
                    rec["status"] = s3_status(res, vdir, rec)
                    recs.append(rec)
            while hr.read(8 << 20):
                pass
            if hr.h.hexdigest() != r["md5sum"]:
                raise RuntimeError(f"MD5 gói sai {hr.h.hexdigest()}")
        except Exception as e:                       # mạng cắt, MD5 sai...: dọn sạch rồi thử lại
            for d in created:
                shutil.rmtree(d, ignore_errors=True)
            if attempt < S3_ATTEMPTS and not stop.is_set():
                time.sleep(2 ** attempt); continue
            pkg_log.write({**info, "status": "failed", "reason": f"{type(e).__name__}: {e}", "attempts": attempt, "finished_at": now()})
            return {**info, "status": "failed"}
        for rec in recs:                             # chỉ ghi bản ghi SAU KHI MD5 gói khớp
            ver_log.write(rec)
            with state_lock:
                state[(rec["arxiv_id"], rec["version"])] = rec
        drive_part("s3_" + Path(key).stem, created, data_root)
        info.update(status="done", extracted=len(recs), seconds=round(time.perf_counter() - t0, 1), attempts=attempt, bytes=int(r["size"]))
        pkg_log.write({**info, "finished_at": now()})
        return info

def arxiv_lane(todo, client, state, ver_log, counters, tag):
    since_part = []
    for pid, v in todo:
        if stop.is_set():
            break
        if ARXIV_BUDGET and counters["arxiv_versions"] >= ARXIV_BUDGET:
            counters["budget_hit"] = True
            break
        rec = process_version(client, pid, v, data_root=data_root, keep_extensions=KEEP)
        rec["finished_at"], rec["source"] = now(), "export"
        ver_log.write(rec)
        with state_lock:
            state[(pid, v)] = rec
        counters["arxiv_versions"] += 1
        if rec["status"] in FINAL_STATUSES:
            since_part.append(version_dir(data_root, pid, v))
        if len(since_part) >= ARXIV_PART_EVERY:
            drive_part(f"ax_{tag}_{counters['arxiv_versions']}", since_part, data_root); drive_sync_work(work); since_part = []
        if counters["arxiv_versions"] % 50 == 0:
            print(f"[arXiv {tag}] {counters['arxiv_versions']} phiên bản")
    drive_part(f"ax_{tag}_end_{counters['arxiv_versions']}", since_part, data_root)

In [ ]:
# 6. Chạy: discovery -> (S3 || arXiv || references) -> lượt vét -> tổng kết (chạy lại để tiếp tục)
summary, found = {}, {}
if RUN:
    data_root.mkdir(parents=True, exist_ok=True); work.mkdir(parents=True, exist_ok=True)
    drive_restore(work, data_root)
    lock = WorkDirLock(work); lock.acquire()
    contact = (_secret("ARXIV_CONTACT_EMAIL") or "").strip()
    client = HttpClient(get_arxiv_limiter(), "HCMUS-IntroDS-Group01-crawler/0.1 (university course project{})".format(f"; mailto:{contact}" if contact else ""))
    ver_log, disc_log, pkg_log = LockedWriter(work / "versions.jsonl"), LockedWriter(work / "discovery.jsonl"), LockedWriter(work / "s3_packages.jsonl")
    sampler = Sampler(str(out)); sampler.start()
    counters = {"arxiv_versions": 0, "budget_hit": False}
    t0, interrupted, s3_results, ref_thread, ex, missing, discovery_s = time.perf_counter(), False, [], None, None, {}, 0.0
    try:
        disc = discover(ids, client, data_root, journal=disc_log.write)
        found, missing, discovery_s = disc.found, disc.missing, disc.elapsed_s
        drive_part("metadata", [data_root / to_folder_name(i) / "metadata.json" for i in found], data_root); drive_sync_work(work)
        print(f"discovery: {len(found)}/{len(ids)} bài, {sum(found.values())} phiên bản, {discovery_s:.0f}s, thiếu {len(missing)}")
        if REFERENCES:
            from src.crawler import references
            ref_thread = threading.Thread(target=references.crawl, args=([i for i in ids if i in found], data_root), daemon=True); ref_thread.start()
        state = load_version_state(work / "versions.jsonl")

        s3_cover, todo_pkgs = set(), []              # (bài, bản mới nhất) mà S3 sẽ phủ
        for r in pkgs:
            a, b = id_key(r["first_item"]), id_key(r["last_item"])
            mine = [f"{a[0]:04d}.{n:05d}" for n in range(a[1], b[1] + 1) if f"{a[0]:04d}.{n:05d}" in found]
            cover = [(p, found[p]) for p in mine if latest_date(read_metadata(data_root, p)) <= pkg_ts(r) - timedelta(days=STALE_MARGIN_DAYS)]
            if cover:
                todo_pkgs.append(r); s3_cover.update(cover)
        pending = lambda: [(i, v) for i in ids for v in range(1, found.get(i, 0) + 1) if not is_done(state.get((i, v)), data_root)]
        todo1 = [x for x in pending() if x not in s3_cover]
        print(f"{sum(found.values())} phiên bản | S3 phủ {len(s3_cover)} ({len(todo_pkgs)} gói) | arXiv lượt 1: {len(todo1)}")

        ex = ThreadPoolExecutor(WORKERS_S3)
        futs = [ex.submit(s3_package, r, found, state, ver_log, pkg_log) for r in todo_pkgs]
        arxiv_lane(todo1, client, state, ver_log, counters, "p1")
        s3_results = [f.result() for f in futs]
        ex.shutdown()
        todo2 = pending()
        print(f"lượt vét (S3 lỗi/thiếu, bản còn lại): {len(todo2)} phiên bản")
        arxiv_lane(todo2, client, state, ver_log, counters, "p2")
        if ref_thread is not None:
            ref_thread.join()
    except BaseException:
        interrupted = True; stop.set()
        if ex is not None:
            ex.shutdown(wait=False, cancel_futures=True)
        raise
    finally:
        sampler.stop.set(); wall = time.perf_counter() - t0
        for w in (ver_log, disc_log, pkg_log):
            w.close()
        if found:
            summary = {"first_id": ids[0], "last_id": ids[-1], "interrupted": interrupted, "budget_hit": counters["budget_hit"], "wall_s": round(wall, 1),
                       "discovery_s": round(discovery_s, 1), "arxiv_requests": client.requests_sent, "arxiv_versions": counters["arxiv_versions"],
                       "min_request_gap_s": round(client.limiter.min_gap_s, 3) if client.limiter.min_gap_s is not None else None,
                       "s3_packages": dict(Counter(x.get("status") for x in s3_results)), "s3_gb": round(sum(x.get("bytes", 0) for x in s3_results) / 1e9, 2),
                       **sampler.stats(), **summarize(ids, found, missing, state, data_root)}
            write_failed_csv(work / "failed.csv", ids, found, missing, state)
            (work / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
        drive_sync_work(work)
        lock.release()
    print(json.dumps(summary, indent=2))
    if counters["budget_hit"]:
        print("Hết ngân sách arXiv của PRESET; chạy lại cell 6 để tiếp tục (không đọc lại gói S3 đã xong).")

In [ ]:
# 7. Kiểm chứng và báo cáo
if summary and not summary["interrupted"]:
    st = load_version_state(work / "versions.jsonl")
    allv = [(i, v) for i in ids for v in range(1, found.get(i, 0) + 1)]
    absent = [x for x in allv if not is_done(st.get(x), data_root)]
    print(f"Phiên bản chưa xong: {len(absent)}/{len(allv)} | theo nguồn: {dict(Counter(st[x].get('source') for x in allv if x in st))}")

    s3_recs = [r for r in st.values() if r.get("source") == "s3" and r["status"] in ("ok", "no_tex")]
    sample = ([r for r in s3_recs if r["status"] == "no_tex"][:1]
              + sorted((r for r in s3_recs if r["status"] == "ok"), key=lambda r: -r["version"])[:max(VERIFY_SAMPLES - 1, 0)])[:VERIFY_SAMPLES]
    bad = []
    for r in sample:                                 # S3 == export, từng byte, qua chính downloader của dự án
        ref = out / "verify" / to_folder_name(r["arxiv_id"]) / f"v{r['version']}"
        dl = download_version(client, r["arxiv_id"], r["version"], ref, KEEP)
        same = dl.status == "ok" and tree(ref) == tree(version_dir(data_root, r["arxiv_id"], r["version"])) and dl.extract.dropped == r["dropped"]
        print(" ", r["arxiv_id"], f"v{r['version']}", r["status"], "IDENTICAL" if same else f"DIFFERENT ({dl.status})")
        if not same:
            bad.append(r["arxiv_id"])
    shutil.rmtree(out / "verify", ignore_errors=True)
    if sample:
        print("Đối chiếu S3 với export:", "PASS" if not bad else f"FAIL {bad}")

    s = summary
    print(f"\nBÁO CÁO: {s['papers_complete']}/{s['ids_requested']} bài đầy đủ (tỷ lệ {s['paper_success_rate']:.1%}) | {s['versions_total']} phiên bản "
          f"| trung bình mỗi bài {s['avg_paper_bytes_before'] or 0:,} -> {s['avg_paper_bytes_after'] or 0:,} B (trước/sau bỏ ảnh)")
    print(f"thời gian {s['wall_s']:.0f}s (discovery {s['discovery_s']:.0f}s) | arXiv {s['arxiv_requests']} request, khoảng cách tối thiểu {s['min_request_gap_s']} s "
          f"| S3 {s['s3_gb']} GB {s['s3_packages']} | RAM đỉnh/TB {s['ram_max_mb']}/{s['ram_mean_mb']} MB | đĩa đỉnh {s['disk_used_max_gb']} GB")
    refs = [json.loads(p.read_text(encoding='utf-8')) for p in data_root.glob("*/references.json")]
    if refs:
        okr = [x for x in refs if x.get("status") == "ok"]
        print(f"references: {len(okr)}/{len(refs)} bài ok | trung bình {sum(x['num_references'] for x in okr) / max(len(okr), 1):.1f} tham chiếu/bài")